In [ ]:
import torch
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision.datasets import ImageFolder
from sklearn.model_selection import train_test_split
from pathlib import Path
from typing import Tuple, List, Optional

# ----------------------------- 自定义数据集包装器 -----------------------------
class SkinDataset(Dataset):
    """自定义数据集，接受样本列表 (path, label) 和 transform"""
    def __init__(self, samples: List[Tuple[str, int]], transform=None):
        self.samples = samples
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        image = Image.open(path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return image, label

# 为避免循环导入，在函数内导入 Image
from PIL import Image

# ----------------------------- 数据集创建与划分 -----------------------------
def get_datasets(
    data_dir: str,
    input_size: int = 224,
    val_ratio: float = 0.2,
    seed: int = 42,
) -> Tuple[Dataset, Dataset, Dataset, List[str], List[int]]:
    """
    加载并划分皮肤病数据集，返回训练、验证、测试数据集、类别名称和完整训练标签。
    
    Args:
        data_dir: 数据集根目录，需包含 Train/ 和 Test/ 子目录
        input_size: 输入图像尺寸 (正方形)
        val_ratio: 从训练集中划分的验证集比例
        seed: 随机种子，保证可复现
        
    Returns:
        train_dataset: 训练集 (带数据增强)
        val_dataset: 验证集 (仅基础预处理)
        test_dataset: 测试集 (仅基础预处理)
        class_names: 类别名称列表 (按 ImageFolder 的索引顺序)
        train_targets: 完整训练集的标签列表 (未划分前，用于计算类别权重)
    """
    train_dir = Path(r"D:\skincode\SkinDisease\SkinDisease\train") 
    test_dir = Path(r"D:\skincode\SkinDisease\SkinDisease\test")
    
    # 1. 加载完整训练集（不带 transform，仅获取样本和标签）
    full_train_dataset = ImageFolder(train_dir, transform=None)
    class_names = full_train_dataset.classes
    # 样本列表: list of (image_path, label)
    samples = full_train_dataset.samples
    targets = full_train_dataset.targets  # 与 samples 顺序一致
    
    # 2. 分层划分训练/验证索引
    train_indices, val_indices = train_test_split(
        range(len(samples)),
        test_size=val_ratio,
        stratify=targets,
        random_state=seed
    )
    
    # 3. 定义 transform
    # 训练集增强 (适度)
    train_transform = transforms.Compose([
        transforms.RandomResizedCrop(input_size),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(degrees=10),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])
    
    # 验证集和测试集：仅调整大小 + 中心裁剪 + 归一化
    val_test_transform = transforms.Compose([
        transforms.Resize(int(input_size * 1.0)),  # 先放大一点再中心裁剪
        transforms.CenterCrop(input_size),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])
    
    # 4. 构建训练集和验证集 (使用自定义 Dataset 以便分别应用 transform)
    train_samples = [samples[i] for i in train_indices]
    val_samples = [samples[i] for i in val_indices]
    
    train_dataset = SkinDataset(train_samples, transform=train_transform)
    val_dataset = SkinDataset(val_samples, transform=val_test_transform)
    
    # 5. 加载测试集，并校验类别一致性
    test_full_dataset = ImageFolder(test_dir, transform=None)
    if test_full_dataset.classes != class_names:
        raise ValueError(
            f"测试集类别与训练集不一致！\n"
            f"训练集类别: {class_names}\n"
            f"测试集类别: {test_full_dataset.classes}\n"
            f"请检查数据目录结构。"
        )
    # 测试集使用相同的 transform（无增强）
    test_dataset = SkinDataset(test_full_dataset.samples, transform=val_test_transform)
    
    # 返回完整训练集标签（用于类别权重计算）
    train_targets = targets  # 原始顺序的标签列表
    
    return train_dataset, val_dataset, test_dataset, class_names, train_targets


# ----------------------------- DataLoader 封装 -----------------------------
def get_dataloaders(
    train_dataset: Dataset,
    val_dataset: Dataset,
    test_dataset: Dataset,
    class_names: List[str],
    batch_size: int = 32,
    num_workers: int = 0,
    pin_memory: bool = True,
) -> Tuple[DataLoader, DataLoader, DataLoader, List[str]]:
    """
    为训练、验证、测试集创建 DataLoader。
    
    Args:
        train_dataset: 训练数据集
        val_dataset: 验证数据集
        test_dataset: 测试数据集
        class_names: 类别名称列表
        batch_size: 批次大小
        num_workers: 数据加载子进程数
        pin_memory: 是否使用 pin_memory (GPU 训练时建议 True)
        
    Returns:
        train_loader, val_loader, test_loader, class_names
    """
    train_loader = DataLoader(
        train_dataset, batch_size=batch_size, shuffle=True,
        num_workers=num_workers, pin_memory=pin_memory
    )
    val_loader = DataLoader(
        val_dataset, batch_size=batch_size, shuffle=False,
        num_workers=num_workers, pin_memory=pin_memory
    )
    test_loader = DataLoader(
        test_dataset, batch_size=batch_size, shuffle=False,
        num_workers=num_workers, pin_memory=pin_memory
    )
    return train_loader, val_loader, test_loader, class_names


# ----------------------------- 示例用法 -----------------------------
if __name__ == "__main__":
    # 参数设置

    DATA_DIR = "./data/skin_disease"
    INPUT_SIZE = 224
    BATCH_SIZE = 32
    VAL_RATIO = 0.2
    
    # 1. 获取数据集
    train_dataset, val_dataset, test_dataset, class_names, train_targets = get_datasets(
        data_dir=DATA_DIR,
        input_size=INPUT_SIZE,
        val_ratio=VAL_RATIO,
        seed=42
    )
    
    print("===== 数据集信息 =====")
    print(f"类别数量: {len(class_names)}")
    print(f"类别名称: {class_names}")
    print(f"训练集大小: {len(train_dataset)}")
    print(f"验证集大小: {len(val_dataset)}")
    print(f"测试集大小: {len(test_dataset)}")
    print(f"原始训练集标签数 (用于类别权重): {len(train_targets)}")
    
    # 2. 创建 DataLoader
    train_loader, val_loader, test_loader, _ = get_dataloaders(
        train_dataset=train_dataset,
        val_dataset=val_dataset,
        test_dataset=test_dataset,
        class_names=class_names,
        batch_size=BATCH_SIZE,
        num_workers=0,
        pin_memory=torch.cuda.is_available()  # 有 GPU 则开启
    )
    
    # 3. 获取一个 batch 示例
    images, labels = next(iter(train_loader))
    print("\n===== 一个 batch 示例 (来自训练集) =====")
    print(f"图像 shape: {images.shape}")   # [B, 3, H, W]
    print(f"标签 shape: {labels.shape}")   # [B]
    # 打印该 batch 中的类别名称
    batch_class_names = [class_names[label] for label in labels]
    print(f"该 batch 包含的类别: {batch_class_names}")

===== 数据集信息 =====
类别数量: 22
类别名称: ['Acne', 'Actinic_Keratosis', 'Benign_tumors', 'Bullous', 'Candidiasis', 'DrugEruption', 'Eczema', 'Infestations_Bites', 'Lichen', 'Lupus', 'Moles', 'Psoriasis', 'Rosacea', 'Seborrh_Keratoses', 'SkinCancer', 'Sun_Sunlight_Damage', 'Tinea', 'Unknown_Normal', 'Vascular_Tumors', 'Vasculitis', 'Vitiligo', 'Warts']
训练集大小: 11118
验证集大小: 2780
测试集大小: 1546
原始训练集标签数 (用于类别权重): 13898

===== 一个 batch 示例 (来自训练集) =====
图像 shape: torch.Size([32, 3, 224, 224])
标签 shape: torch.Size([32])
该 batch 包含的类别: ['Rosacea', 'Rosacea', 'DrugEruption', 'Candidiasis', 'Actinic_Keratosis', 'Warts', 'Bullous', 'Tinea', 'Bullous', 'Acne', 'Sun_Sunlight_Damage', 'Unknown_Normal', 'Actinic_Keratosis', 'Psoriasis', 'Eczema', 'Actinic_Keratosis', 'Benign_tumors', 'Lupus', 'Actinic_Keratosis', 'DrugEruption', 'Benign_tumors', 'Eczema', 'Unknown_Normal', 'Benign_tumors', 'Warts', 'Benign_tumors', 'DrugEruption', 'SkinCancer', 'Unknown_Normal', 'Unknown_Normal', 'Eczema', 'Benign_tumors']


In [2]:
# model_efficientnet_b0.py
import torch
import torch.nn as nn
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights

def get_efficientnet_b0_baseline(
    num_classes: int = 22,
    pretrained: bool = True,
    freeze_backbone: bool = True,
) -> torch.nn.Module:
    """
    创建 EfficientNet-B0 基线模型。
    
    Args:
        num_classes: 分类数量，默认 22
        pretrained: 是否加载 ImageNet 预训练权重
        freeze_backbone: 是否冻结 backbone（除分类头外的所有层）
    
    Returns:
        model: 修改分类头后的模型
    """
    if pretrained:
        weights = EfficientNet_B0_Weights.DEFAULT
        model = efficientnet_b0(weights=weights)
    else:
        model = efficientnet_b0(weights=None)
    
    # 修改分类头：efficientnet_b0 的 classifier 是一个 Sequential，最后一个 Linear 需要替换
    in_features = model.classifier[1].in_features
    model.classifier[1] = nn.Linear(in_features, num_classes)
    
    if freeze_backbone:
        # 冻结除 classifier 之外的所有参数
        for name, param in model.named_parameters():
            if "classifier" not in name:
                param.requires_grad = False
    return model

# ----------------------------- 示例用法 -----------------------------
if __name__ == "__main__":
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = get_efficientnet_b0_baseline(num_classes=22, pretrained=True, freeze_backbone=True)
    model.to(device)
    
    # 打印模型结构（需安装 torchinfo）
    try:
        from torchinfo import summary
        summary(model, input_size=(1, 3, 224, 224), device=device)
    except ImportError:
        print("请安装 torchinfo: pip install torchinfo")
        print(model)
    
    # 示例：解冻最后两层分类头之前的层（解冻 block 最后一层）
    # for name, param in model.features[-2:].named_parameters():
    #     param.requires_grad = True
    # 示例：解冻全部层
    # for param in model.parameters():
    #     param.requires_grad = True

In [3]:
import torch
import torch.nn as nn
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights

class SEBlock(nn.Module):
    """Squeeze-and-Excitation 模块"""
    def __init__(self, channels, reduction=16):
        super(SEBlock, self).__init__()
        self.global_avg_pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Linear(channels, channels // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(channels // reduction, channels, bias=False),
            nn.Sigmoid()
        )
    
    def forward(self, x):
        B, C, H, W = x.shape
        y = self.global_avg_pool(x).view(B, C)
        y = self.fc(y).view(B, C, 1, 1)
        return x * y.expand_as(x)

class EfficientNetWithSE(nn.Module):
    """在 EfficientNet-B0 中间插入 SE 模块"""
    def __init__(self, num_classes=22, reduction=16, pretrained=True):
        super(EfficientNetWithSE, self).__init__()
        # 加载原始模型（不含 SE 模块）
        if pretrained:
            weights = EfficientNet_B0_Weights.DEFAULT
            self.backbone = efficientnet_b0(weights=weights)
        else:
            self.backbone = efficientnet_b0(weights=None)
        
        # 获取特征提取层（features）和分类头（classifier）
        self.features = self.backbone.features
        self.avgpool = self.backbone.avgpool
        original_classifier = self.backbone.classifier
        in_features = original_classifier[1].in_features
        
        # 修改分类头
        self.classifier = nn.Sequential(
            original_classifier[0],   # Dropout
            nn.Linear(in_features, num_classes)
        )
        
        # 动态获取 SE 模块的输入通道数（对应 features[7]）
        dummy = torch.randn(1, 3, 224, 224)
        with torch.no_grad():
            for i, block in enumerate(self.features):
                dummy = block(dummy)
                if i == 7:  # 对应第8个MBConv块后
                    se_channels = dummy.shape[1]
                    break
        self.se = SEBlock(channels=se_channels, reduction=reduction)
        
        # 冻结 backbone 的预训练权重
        self._freeze_backbone()
    
    def _freeze_backbone(self):
        """冻结原始 backbone 的所有参数"""
        for param in self.backbone.parameters():
            param.requires_grad = False
        # 确保新添加的 SE 模块可训练
        for param in self.se.parameters():
            param.requires_grad = True
        # 新的分类头可训练
        for param in self.classifier.parameters():
            param.requires_grad = True
    
    def forward(self, x):
        """前向传播，在 features 的第8个块（索引7）之后插入 SE"""
        for i, block in enumerate(self.features):
            x = block(x)
            if i == 7:  # 经过第8个块后插入 SE
                x = self.se(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.classifier(x)
        return x

# ----------------------------- 验证权重加载 -----------------------------
def check_weights_loading(model, pretrained=True):
    if pretrained:
        pure_model = efficientnet_b0(weights=EfficientNet_B0_Weights.DEFAULT)
        pure_state = pure_model.state_dict()
        current_state = model.backbone.state_dict()
        missing, unexpected = [], []
        for k in pure_state:
            if k not in current_state:
                missing.append(k)
        for k in current_state:
            if k not in pure_state:
                unexpected.append(k)
        print("Backbone 权重验证：")
        print(f"  缺失键: {missing}")
        print(f"  意外键: {unexpected}")
        if not missing and not unexpected:
            print("✅ 所有 backbone 权重均正确加载预训练值")

if __name__ == "__main__":
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = EfficientNetWithSE(num_classes=22, reduction=16, pretrained=True)
    model.to(device)
    
    # 验证权重加载
    check_weights_loading(model)
    
    # 测试前向传播
    dummy_input = torch.randn(1, 3, 224, 224).to(device)
    output = model(dummy_input)
    print(f"前向传播成功！输出 shape: {output.shape}")  # 输出应为 [1, 22]

Backbone 权重验证：
  缺失键: []
  意外键: []
✅ 所有 backbone 权重均正确加载预训练值
前向传播成功！输出 shape: torch.Size([1, 22])


In [4]:
# model_vit.py
import torch
import torch.nn as nn
from torchvision.models import vit_b_16, ViT_B_16_Weights

def get_vit_baseline(num_classes: int = 22, pretrained: bool = True) -> nn.Module:
    """
    创建 Vision Transformer ViT-B/16 基线模型。
    
    Args:
        num_classes: 分类数量
        pretrained: 是否加载 ImageNet 预训练权重
    
    Returns:
        model: 修改分类头后的模型
    """
    if pretrained:
        weights = ViT_B_16_Weights.DEFAULT
        model = vit_b_16(weights=weights)
    else:
        model = vit_b_16(weights=None)
    
    # 修改分类头：ViT 的 heads.head 是一个 Linear
    in_features = model.heads.head.in_features
    model.heads.head = nn.Linear(in_features, num_classes)
    
    return model

def get_vit_transforms(input_size=224):
    """获取官方推荐的预处理变换（用于数据加载）"""
    weights = ViT_B_16_Weights.DEFAULT
    return weights.transforms()  # 包含 Resize, ToTensor, Normalize

# ----------------------------- 示例用法 -----------------------------
if __name__ == "__main__":
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = get_vit_baseline(num_classes=22, pretrained=True)
    model.to(device)
    
    # 打印模型结构
    try:
        from torchinfo import summary
        summary(model, input_size=(1, 3, 224, 224), device=device)
    except ImportError:
        print(model)
    
    # 如果需要切换到 384×384，需要加载对应的权重（注意 torchvision 版本）
    # try:
    #     from torchvision.models import vit_b_16, ViT_B_16_Weights
    #     weights_384 = ViT_B_16_Weights.IMAGENET1K_SWAG_E2E_V1  # 如果存在
    #     model_384 = vit_b_16(weights=weights_384)
    # except (ImportError, AttributeError):
    #     print("当前 torchvision 版本不支持 384 输入，请升级或使用默认 224")

In [5]:
# loss_functions.py
import torch
import torch.nn as nn
import torch.nn.functional as F
from typing import List, Optional, Union

def compute_class_weights(train_targets: List[int]) -> torch.Tensor:
    """
    计算类别权重，与类别频率成反比。
    
    Args:
        train_targets: 原始训练集的标签列表（每个样本的类别索引）
    
    Returns:
        class_weights: shape (num_classes,)，每个类别的权重
    """
    targets_tensor = torch.tensor(train_targets, dtype=torch.long)
    num_classes = targets_tensor.max().item() + 1
    class_counts = torch.bincount(targets_tensor, minlength=num_classes).float()
    # 避免除零：若某类样本数为0，权重设为0（实际不会发生，但安全起见）
    class_counts[class_counts == 0] = 1.0
    total = class_counts.sum()
    class_weights = total / (num_classes * class_counts)
    return class_weights

class FocalLoss(nn.Module):
    """
    Focal Loss 实现，适用于多分类。
    公式: FL(p_t) = -alpha_t * (1 - p_t)^gamma * log(p_t)
    """
    def __init__(
        self,
        alpha: Optional[Union[float, torch.Tensor]] = None,
        gamma: float = 2.0,
        reduction: str = 'mean'
    ):
        """
        Args:
            alpha: 类别权重。可以是标量（所有类别相同）或张量（每个类别单独权重）。
            gamma: 聚焦参数，默认2.0。
            reduction: 'none', 'mean', 'sum'
        """
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, inputs: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        """
        Args:
            inputs: 模型输出 logits，shape (N, C)
            targets: 真实标签，shape (N,)
        """
        # 计算交叉熵损失（不缩减，保留每个样本的损失）
        ce_loss = F.cross_entropy(inputs, targets, reduction='none')
        # 计算 p_t：模型对正确类别的预测概率
        pt = torch.exp(-ce_loss)  # 因为 ce_loss = -log(pt)，所以 pt = exp(-ce_loss)
        # 计算 focal loss 调制因子 (1 - pt)^gamma
        focal_weight = (1 - pt) ** self.gamma
        # 应用 alpha 权重
        if self.alpha is not None:
            # 若 alpha 是张量，则按类别取权重
            if isinstance(self.alpha, torch.Tensor):
                alpha_t = self.alpha[targets]
            else:
                alpha_t = self.alpha
            focal_weight = alpha_t * focal_weight
        loss = focal_weight * ce_loss
        # 缩减
        if self.reduction == 'mean':
            return loss.mean()
        elif self.reduction == 'sum':
            return loss.sum()
        else:
            return loss

def get_loss_function(
    loss_type: str = 'weighted_ce',
    train_targets: Optional[List[int]] = None,
    alpha: Optional[Union[float, torch.Tensor]] = None,
    gamma: float = 2.0,
    device: Optional[torch.device] = None
) -> nn.Module:
    """
    根据指定类型返回损失函数。
    
    Args:
        loss_type: 支持 'ce', 'weighted_ce', 'focal'
        train_targets: 原始训练集标签列表（用于计算类别权重，仅在 loss_type 需要时提供）
        alpha: Focal Loss 的 alpha 参数（可以是标量或类别权重张量）
        gamma: Focal Loss 的 gamma 参数
        device: 将类别权重移至的设备（如 'cuda' 或 'cpu'）
    
    Returns:
        criterion: 损失函数模块
    """
    if loss_type == 'ce':
        criterion = nn.CrossEntropyLoss()
    elif loss_type == 'weighted_ce':
        if train_targets is None:
            raise ValueError("weighted_ce 需要提供 train_targets 参数来计算类别权重")
        class_weights = compute_class_weights(train_targets)
        if device is not None:
            class_weights = class_weights.to(device)
        criterion = nn.CrossEntropyLoss(weight=class_weights)
    elif loss_type == 'focal':
        # 处理 alpha：如果未提供且 train_targets 存在，则自动计算类别权重作为 alpha
        if alpha is None and train_targets is not None:
            alpha = compute_class_weights(train_targets)
            if device is not None and isinstance(alpha, torch.Tensor):
                alpha = alpha.to(device)
        criterion = FocalLoss(alpha=alpha, gamma=gamma)
    else:
        raise ValueError(f"不支持的损失类型: {loss_type}，可选 'ce', 'weighted_ce', 'focal'")
    return criterion

# ----------------------------- 示例用法 -----------------------------
if __name__ == "__main__":
    # 模拟 train_targets（假设有 1000 个样本，10 个类别，类别不平衡）
    import random
    random.seed(42)
    train_targets = [random.randint(0, 9) for _ in range(1000)]
    # 人为制造不平衡：让类别0出现很少
    for _ in range(10):
        train_targets.append(0)
    
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    # 计算类别权重并查看
    weights = compute_class_weights(train_targets)
    print("类别权重:", weights)
    
    # 创建损失函数
    criterion = get_loss_function(
        loss_type='weighted_ce',
        train_targets=train_targets,
        device=device
    )
    criterion.to(device)  # 将损失函数的权重参数移至 GPU（CrossEntropyLoss 的 weight 会随之移动）
    print(f"损失函数类型: {type(criterion)}，设备: {next(criterion.parameters()).device if list(criterion.parameters()) else 'CPU'}")
    
    # 模拟一个 batch 数据
    batch_size = 4
    num_classes = 10
    logits = torch.randn(batch_size, num_classes).to(device)
    labels = torch.tensor([0, 5, 2, 0]).to(device)
    
    loss = criterion(logits, labels)
    print(f"加权交叉熵损失值: {loss.item()}")
    
    # 使用 Focal Loss（自动计算 alpha 为类别权重）
    focal_criterion = get_loss_function(
        loss_type='focal',
        train_targets=train_targets,
        gamma=2.0,
        device=device
    )
    focal_loss = focal_criterion(logits, labels)
    print(f"Focal Loss 值: {focal_loss.item()}")
    
    # 使用普通交叉熵
    ce_criterion = get_loss_function(loss_type='ce')
    ce_loss = ce_criterion(logits, labels)
    print(f"普通交叉熵损失值: {ce_loss.item()}")

类别权重: tensor([1.0000, 0.9352, 1.1882, 0.8707, 0.8938, 1.2317, 0.9902, 0.9619, 0.9806,
        1.0632])
损失函数类型: <class 'torch.nn.modules.loss.CrossEntropyLoss'>，设备: CPU
加权交叉熵损失值: 2.3190159797668457
Focal Loss 值: 2.0811169147491455
普通交叉熵损失值: 2.26328444480896


In [6]:
# train.py
import os
import random
import csv
from pathlib import Path
from datasets import get_datasets, get_dataloaders
from models import get_efficientnet_b0_baseline, EfficientNetWithSE
from loss_functions import get_loss_function
import numpy as np
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import DataLoader
from tqdm import tqdm
import matplotlib.pyplot as plt

# 假设以下模块已经实现并位于同一目录或可导入路径
# from get_datasets import get_datasets
# from get_dataloaders import get_dataloaders
# from model_efficientnet_b0 import get_efficientnet_b0_baseline
# from loss_functions import get_loss_function

# ----------------------------- 随机种子固定 -----------------------------
def set_seed(seed: int = 42):
    """固定所有随机种子，确保可复现性"""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# ----------------------------- Checkpoint 保存 -----------------------------
def save_checkpoint(state, class_names, filename):
    """
    保存检查点，必须显式传入 class_names。
    state: 包含 model_state_dict, optimizer_state_dict, epoch, best_val_acc 等
    class_names: 类别名称列表
    filename: 保存路径
    """
    state['class_names'] = class_names
    torch.save(state, filename)

def load_checkpoint(filename, model, optimizer=None):
    """
    加载检查点，返回 epoch, best_val_acc, class_names。
    若 optimizer 提供则加载其状态。
    """
    checkpoint = torch.load(filename, map_location='cpu')
    model.load_state_dict(checkpoint['model_state_dict'])
    if optimizer is not None:
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    epoch = checkpoint.get('epoch', 0)
    best_val_acc = checkpoint.get('best_val_acc', 0.0)
    class_names = checkpoint.get('class_names', None)
    return epoch, best_val_acc, class_names

# ----------------------------- 训练一个 epoch -----------------------------
def train_one_epoch(model, loader, criterion, optimizer, device, epoch, log_interval=50):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    pbar = tqdm(loader, desc=f"Epoch {epoch+1} [Train]", leave=False)
    for batch_idx, (inputs, targets) in enumerate(pbar):
        inputs, targets = inputs.to(device), targets.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * inputs.size(0)
        _, predicted = outputs.max(1)
        total += targets.size(0)
        correct += predicted.eq(targets).sum().item()
        
        # 更新进度条显示
        pbar.set_postfix({"Loss": f"{loss.item():.4f}", "Acc": f"{100.*correct/total:.2f}%"})
    epoch_loss = running_loss / total
    epoch_acc = 100. * correct / total
    return epoch_loss, epoch_acc

@torch.no_grad()
def validate(model, loader, criterion, device, epoch, desc="Val"):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    pbar = tqdm(loader, desc=f"Epoch {epoch+1} [{desc}]", leave=False)
    for inputs, targets in pbar:
        inputs, targets = inputs.to(device), targets.to(device)
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        running_loss += loss.item() * inputs.size(0)
        _, predicted = outputs.max(1)
        total += targets.size(0)
        correct += predicted.eq(targets).sum().item()
        pbar.set_postfix({"Loss": f"{loss.item():.4f}", "Acc": f"{100.*correct/total:.2f}%"})
    epoch_loss = running_loss / total
    epoch_acc = 100. * correct / total
    return epoch_loss, epoch_acc

# ----------------------------- 绘制训练曲线 -----------------------------
def plot_curves(train_losses, val_losses, train_accs, val_accs, best_epoch, save_path="figures/training_curves.png"):
    Path("figures").mkdir(parents=True, exist_ok=True)
    epochs = range(1, len(train_losses)+1)
    
    plt.figure(figsize=(12, 5))
    plt.subplot(1, 2, 1)
    plt.plot(epochs, train_losses, 'b-', label='Train Loss')
    plt.plot(epochs, val_losses, 'r-', label='Val Loss')
    plt.scatter(best_epoch+1, val_losses[best_epoch], color='g', s=100, label='Best Model (Val Acc)')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title('Loss Curves')
    plt.legend()
    plt.grid(True)
    
    plt.subplot(1, 2, 2)
    plt.plot(epochs, train_accs, 'b-', label='Train Acc')
    plt.plot(epochs, val_accs, 'r-', label='Val Acc')
    plt.scatter(best_epoch+1, val_accs[best_epoch], color='g', s=100, label='Best Model')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy (%)')
    plt.title('Accuracy Curves')
    plt.legend()
    plt.grid(True)
    
    plt.tight_layout()
    plt.savefig(save_path, dpi=150)
    plt.close()
    print(f"训练曲线已保存至 {save_path}")

def save_log(epoch, train_loss, val_loss, train_acc, val_acc, log_path="training_log.csv"):
    """将指标保存到 CSV 文件"""
    file_exists = Path(log_path).exists()
    with open(log_path, 'a', newline='') as f:
        writer = csv.writer(f)
        if not file_exists:
            writer.writerow(['epoch', 'train_loss', 'val_loss', 'train_acc', 'val_acc'])
        writer.writerow([epoch+1, train_loss, val_loss, train_acc, val_acc])

# ----------------------------- 主训练流程 -----------------------------
def main():
    # ---------- 配置 ----------
    DATA_DIR = r"D:\skincode\SkinDisease\SkinDisease"
    INPUT_SIZE = 224
    BATCH_SIZE = 32
    VAL_RATIO = 0.2
    NUM_CLASSES = 22
    SEED = 42
    EPOCHS = 30
    LEARNING_RATE = 1e-3
    WEIGHT_DECAY = 1e-4
    RESUME = False          # 是否从断点恢复
    CHECKPOINT_PATH = "checkpoint.pth"
    BEST_MODEL_PATH = "best_efficientnet_b0.pth"
    
    set_seed(SEED)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"使用设备: {device}")
    
    # ---------- 数据加载 ----------
    # 假设 get_datasets 和 get_dataloaders 已经定义

    train_dataset, val_dataset, test_dataset, class_names, train_targets = get_datasets(
        data_dir=DATA_DIR,
        input_size=INPUT_SIZE,
        val_ratio=VAL_RATIO,
        seed=SEED
    )
    train_loader, val_loader, test_loader, _ = get_dataloaders(
        train_dataset=train_dataset,
        val_dataset=val_dataset,
        test_dataset=test_dataset,
        class_names=class_names,
        batch_size=BATCH_SIZE,
        num_workers=0,
        pin_memory=True
    )
    
    # ---------- 模型、损失、优化器 ----------
    model = get_efficientnet_b0_baseline(num_classes=NUM_CLASSES, pretrained=True, freeze_backbone=False)
    model.to(device)
    
    criterion = get_loss_function(loss_type='weighted_ce', train_targets=train_targets, device=device)
    
    optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=0)
    
    # ---------- 断点恢复 ----------
    start_epoch = 0
    best_val_acc = 0.0
    if RESUME and Path(CHECKPOINT_PATH).exists():
        print(f"加载断点: {CHECKPOINT_PATH}")
        start_epoch, best_val_acc, _ = load_checkpoint(CHECKPOINT_PATH, model, optimizer)
        # 注意: scheduler 需要根据 epoch 恢复步数，可自行设置 scheduler.last_epoch = start_epoch - 1
        if start_epoch > 0:
            scheduler.last_epoch = start_epoch - 1
        print(f"恢复至 epoch {start_epoch}, 最佳验证准确率 {best_val_acc:.2f}%")
    
    # ---------- 记录列表 ----------
    train_losses, val_losses = [], []
    train_accs, val_accs = [], []
    
    # ---------- 训练循环 ----------
    for epoch in range(start_epoch, EPOCHS):
        # 训练
        train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device, epoch)
        # 验证
        val_loss, val_acc = validate(model, val_loader, criterion, device, epoch, desc="Val")
        
        # 记录
        train_losses.append(train_loss)
        val_losses.append(val_loss)
        train_accs.append(train_acc)
        val_accs.append(val_acc)
        save_log(epoch, train_loss, val_loss, train_acc, val_acc)
        
        # 打印
        print(f"Epoch {epoch+1}/{EPOCHS} | Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}% | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.2f}%")
        
        # 更新学习率
        scheduler.step()
        
        # 保存最佳模型
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_epoch = epoch
            save_checkpoint({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'best_val_acc': best_val_acc,
            }, class_names, BEST_MODEL_PATH)
            print(f"  -> 新最佳模型已保存 (Val Acc: {val_acc:.2f}%)")
        
        # 保存断点（每个 epoch 结束）
        save_checkpoint({
            'epoch': epoch + 1,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'best_val_acc': best_val_acc,
        }, class_names, CHECKPOINT_PATH)
    
    # ---------- 绘制曲线 ----------
    # 找到最佳模型对应的 epoch（索引）
    best_epoch_idx = val_accs.index(max(val_accs))
    plot_curves(train_losses, val_losses, train_accs, val_accs, best_epoch_idx)
    
    print(f"\n训练完成。最佳验证准确率: {best_val_acc:.2f}% (Epoch {best_epoch_idx+1})")
    print(f"最佳模型已保存至 {BEST_MODEL_PATH}")
    print(f"训练日志已保存至 training_log.csv")

if __name__ == "__main__":
    main()

使用设备: cpu


Epoch 1 [Train]:   0%|          | 0/348 [00:00<?, ?it/s]d:\skincode\code\.venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch 1/30 | Train Loss: 2.3021 | Train Acc: 32.44% | Val Loss: 1.9051 | Val Acc: 42.95%
  -> 新最佳模型已保存 (Val Acc: 42.95%)


Epoch 2/30 | Train Loss: 1.9179 | Train Acc: 42.89% | Val Loss: 1.6174 | Val Acc: 51.12%
  -> 新最佳模型已保存 (Val Acc: 51.12%)


Epoch 3/30 | Train Loss: 1.7319 | Train Acc: 48.01% | Val Loss: 1.5835 | Val Acc: 53.88%
  -> 新最佳模型已保存 (Val Acc: 53.88%)


...

Epoch 26/30 | Train Loss: 0.3110 | Train Acc: 89.48% | Val Loss: 1.2044 | Val Acc: 76.73%
  -> 新最佳模型已保存 (Val Acc: 76.73%)


Epoch 27/30 | Train Loss: 0.3060 | Train Acc: 89.98% | Val Loss: 1.1691 | Val Acc: 77.19%
  -> 新最佳模型已保存 (Val Acc: 77.19%)


Epoch 28/30 | Train Loss: 0.3042 | Train Acc: 90.06% | Val Loss: 1.1716 | Val Acc: 77.16%


Epoch 29/30 | Train Loss: 0.3088 | Train Acc: 89.89% | Val Loss: 1.1834 | Val Acc: 77.81%
  -> 新最佳模型已保存 (Val Acc: 77.81%)


Epoch 30/30 | Train Loss: 0.2977 | Train Acc: 90.05% | Val Loss: 1.1798 | Val Acc: 77.16%
训练曲线已保存至 figures/training_curves.png

训练完成。最佳验证准确率: 77.81% (Epoch 29)
最佳模型已保存至 best_efficientnet_b0.pth
训练日志已保存至 training_log.csv


In [7]:
# utils.py
import torch
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix
from typing import Tuple, List

@torch.no_grad()
def collect_predictions(model, loader, device):
    """
    遍历 DataLoader，收集所有真实标签和预测标签。
    
    Args:
        model: 模型
        loader: DataLoader
        device: 设备
    
    Returns:
        y_true: 真实标签列表
        y_pred: 预测标签列表
    """
    model.eval()
    y_true, y_pred = [], []
    for inputs, targets in loader:
        inputs = inputs.to(device)
        outputs = model(inputs)
        _, preds = outputs.max(1)
        y_true.extend(targets.cpu().numpy())
        y_pred.extend(preds.cpu().numpy())
    return y_true, y_pred

def compute_overall_metrics(y_true, y_pred):
    """
    计算整体指标：Accuracy, Macro/Weighted Precision, Recall, F1.
    
    Returns:
        dict: {
            'accuracy': float,
            'macro_precision': float,
            'macro_recall': float,
            'macro_f1': float,
            'weighted_precision': float,
            'weighted_recall': float,
            'weighted_f1': float
        }
    """
    acc = accuracy_score(y_true, y_pred)
    macro_p, macro_r, macro_f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average='macro', zero_division=0
    )
    weighted_p, weighted_r, weighted_f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average='weighted', zero_division=0
    )
    return {
        'accuracy': acc,
        'macro_precision': macro_p,
        'macro_recall': macro_r,
        'macro_f1': macro_f1,
        'weighted_precision': weighted_p,
        'weighted_recall': weighted_r,
        'weighted_f1': weighted_f1
    }

def compute_specificity_nvp(cm, class_idx):
    """
    根据混淆矩阵计算指定类别的特异性（Specificity）和阴性预测值（NPV）。
    
    Args:
        cm: 混淆矩阵 (n_classes, n_classes)，行=真实，列=预测
        class_idx: 类别索引
    
    Returns:
        specificity: TN / (TN + FP)
        npv: TN / (TN + FN)
    """
    # 真阴性: 所有不是该类且预测也不是该类的样本数
    tn = cm.sum() - (cm[class_idx, :].sum() + cm[:, class_idx].sum() - cm[class_idx, class_idx])
    fp = cm[:, class_idx].sum() - cm[class_idx, class_idx]
    fn = cm[class_idx, :].sum() - cm[class_idx, class_idx]
    tp = cm[class_idx, class_idx]
    
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    npv = tn / (tn + fn) if (tn + fn) > 0 else 0.0
    return specificity, npv

In [10]:
# evaluate.py
import csv
import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_fscore_support



# ----------------------------- 配置 -----------------------------
DATA_DIR = r"D:\skincode\SkinDisease\SkinDisease"
INPUT_SIZE = 224
BATCH_SIZE = 32
VAL_RATIO = 0.2
NUM_CLASSES = 22
SEED = 42
CHECKPOINT_PATH = "best_efficientnet_b0.pth"
OUTPUT_DIR = Path("figures")
OUTPUT_DIR.mkdir(exist_ok=True)

# ----------------------------- 主评估流程 -----------------------------
def main():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"使用设备: {device}")

    # 1. 加载测试集 DataLoader（复用之前的数据划分）
    # 注意：此处需要根据实际项目结构重建 test_loader
    # 假设 get_datasets 和 get_dataloaders 已经定义
    _, _, test_dataset, class_names, _ = get_datasets(
        data_dir=DATA_DIR,
        input_size=INPUT_SIZE,
        val_ratio=VAL_RATIO,
        seed=SEED
    )
    from torch.utils.data import DataLoader

    test_loader = DataLoader(
        test_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=0,  # CPU必须设0
        pin_memory=False
    )

    # 2. 加载模型
    model = get_efficientnet_b0_baseline(num_classes=NUM_CLASSES, pretrained=False, freeze_backbone=False)
    checkpoint = torch.load(CHECKPOINT_PATH, map_location='cpu')
    model.load_state_dict(checkpoint['model_state_dict'])
    model.to(device)
    # 从 checkpoint 中获取 class_names（确保一致性）
    if 'class_names' in checkpoint:
        class_names = checkpoint['class_names']
    print(f"模型加载完成，类别数量: {len(class_names)}")

    # 3. 收集预测结果
    y_true, y_pred = collect_predictions(model, test_loader, device)
    print(f"测试集样本数: {len(y_true)}")

    # 4. 计算整体指标并保存
    overall_metrics = compute_overall_metrics(y_true, y_pred)
    with open("metrics.csv", "w", newline='') as f:
        writer = csv.writer(f)
        writer.writerow(["metric", "value"])
        for k, v in overall_metrics.items():
            writer.writerow([k, f"{v:.4f}"])
    print("\n整体指标已保存至 metrics.csv")
    for k, v in overall_metrics.items():
        print(f"{k}: {v:.4f}")

    # 5. 计算每个类别的指标并保存 per_class_metrics.csv
    per_class = precision_recall_fscore_support(y_true, y_pred, average=None, zero_division=0)
    supports = np.bincount(y_true, minlength=len(class_names))
    with open("per_class_metrics.csv", "w", newline='', encoding='utf-8') as f:
        writer = csv.writer(f)
        writer.writerow(["class_name", "precision", "recall", "f1", "support"])
        for i, name in enumerate(class_names):
            writer.writerow([
                name,
                f"{per_class[0][i]:.4f}",
                f"{per_class[1][i]:.4f}",
                f"{per_class[2][i]:.4f}",
                supports[i]
            ])
    print("每个类别的指标已保存至 per_class_metrics.csv")

    # 6. 混淆矩阵可视化
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(12, 10))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names)
    plt.title('Confusion Matrix on Test Set')
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.xticks(rotation=45, ha='right')
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "confusion_matrix.png", dpi=150)
    plt.close()
    print(f"混淆矩阵已保存至 {OUTPUT_DIR / 'confusion_matrix.png'}")

    # 7. 针对两个关键类别计算特异性与 NPV
    # 假设关键类别名称为 'Skin Cancer' 和 'Actinic Keratosis'，请根据实际 class_names 调整
    # 这里以索引为例（通常皮肤癌类别可能叫 'mel' 或 'bcc'，需根据数据集修改）
    # 用户可根据实际类别名修改下面的映射
    key_class_names = ['Actinic_Keratosis', 'Benign_tumors']  # 示例：黑色素瘤和光化性角化病
    key_indices = []
    for name in key_class_names:
        if name in class_names:
            key_indices.append(class_names.index(name))
        else:
            print(f"警告：类别 '{name}' 不在 class_names 中，跳过计算")
    if len(key_indices) == 2:
        spec1, npv1 = compute_specificity_nvp(cm, key_indices[0])
        spec2, npv2 = compute_specificity_nvp(cm, key_indices[1])
        print("\n===== 关键疾病指标 =====")
        print(f"{class_names[key_indices[0]]}:")
        print(f"  Sensitivity (Recall): {per_class[1][key_indices[0]]:.4f}")
        print(f"  Specificity: {spec1:.4f}")
        print(f"  NPV: {npv1:.4f}")
        print(f"{class_names[key_indices[1]]}:")
        print(f"  Sensitivity (Recall): {per_class[1][key_indices[1]]:.4f}")
        print(f"  Specificity: {spec2:.4f}")
        print(f"  NPV: {npv2:.4f}")
        
        # 写入额外的关键指标文件
        with open("key_disease_metrics.csv", "w", newline='') as f:
            writer = csv.writer(f)
            writer.writerow(["class", "sensitivity", "specificity", "npv"])
            writer.writerow([class_names[key_indices[0]], per_class[1][key_indices[0]], spec1, npv1])
            writer.writerow([class_names[key_indices[1]], per_class[1][key_indices[1]], spec2, npv2])
        
        # 临床解释
        print("\n临床解读:")
        print("- 对于恶性皮肤病（如黑色素瘤），高灵敏度（召回率）意味着漏诊率低，能最大程度避免延误治疗。")
        print("- 高 NPV 意味着如果模型预测为阴性（正常），实际为阴性的概率很高，有助于减少不必要的活检。")
        print("- 特异性衡量正确排除非疾病的能力，在筛查场景中可减少过度治疗。")
    else:
        print("未找到指定的关键类别，跳过特异性/NPV计算。")

if __name__ == "__main__":
    main()

使用设备: cpu
模型加载完成，类别数量: 22
测试集样本数: 1546

整体指标已保存至 metrics.csv
accuracy: 0.7652
macro_precision: 0.7378
macro_recall: 0.7384
macro_f1: 0.7351
weighted_precision: 0.7694
weighted_recall: 0.7652
weighted_f1: 0.7646
每个类别的指标已保存至 per_class_metrics.csv
混淆矩阵已保存至 figures\confusion_matrix.png

===== 关键疾病指标 =====
Actinic_Keratosis:
  Sensitivity (Recall): 0.7952
  Specificity: 0.9863
  NPV: 0.9884
Benign_tumors:
  Sensitivity (Recall): 0.7851
  Specificity: 0.9768
  NPV: 0.9817

临床解读:
- 对于恶性皮肤病（如黑色素瘤），高灵敏度（召回率）意味着漏诊率低，能最大程度避免延误治疗。
- 高 NPV 意味着如果模型预测为阴性（正常），实际为阴性的概率很高，有助于减少不必要的活检。
- 特异性衡量正确排除非疾病的能力，在筛查场景中可减少过度治疗。
